In [1]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt

In [3]:
import os
os.getcwd()


'C:\\Users\\gsrij\\Assignment2\\Q1_UsedCars'

In [4]:
df = pd.read_csv("train.csv")
df.head()


,Unnamed: 0,Name,Location,Year,Kilometers_Driven,Fuel_Type,Transmission,Owner_Type,Mileage,Engine,Power,Seats,New_Price,Price
0,1,Hyundai Creta 1.6 CRDi SX Option,Pune,2015,41000,Diesel,Manual,First,19.67 kmpl,1582 CC,126.2 bhp,5.0,NaN,12.50
1,2,Honda Jazz V,Chennai,2011,46000,Petrol,Manual,First,13 km/kg,1199 CC,88.7 bhp,5.0,8.61 Lakh,4.50
2,3,Maruti Ertiga VDI,Chennai,2012,87000,Diesel,Manual,First,20.77 kmpl,1248 CC,88.76 bhp,7.0,NaN,6.00
3,4,Audi A4 New 2.0 TDI Multitronic,Coimbatore,2013,40670,Diesel,Automatic,Second,15.2 kmpl,1968 CC,140.8 bhp,5.0,NaN,17.74
4,6,Nissan Micra Diesel XV,Jaipur,2013,86999,Diesel,Manual,First,23.08 kmpl,1461 CC,63.1 bhp,5.0,NaN,3.50


In [5]:
df.isnull().sum()


Unnamed: 0              0
Name                    0
Location                0
Year                    0
Kilometers_Driven       0
Fuel_Type               0
Transmission            0
Owner_Type              0
Mileage                 2
Engine                 36
Power                  36
Seats                  38
New_Price            5032
Price                   0
dtype: int64

In [8]:
for col in df.columns:
    if df[col].dtype != "object":
        df[col] = df[col].fillna(df[col].median())
    else:
        df[col] = df[col].fillna(df[col].mode()[0])


In [9]:
df['Mileage'] = df['Mileage'].str.replace(" kmpl","",regex=False)\
                             .str.replace(" km/l","",regex=False)\
                             .str.replace(" km/L","",regex=False)\
                             .str.replace(" km/kg","",regex=False)\
                             .str.strip()

df['Mileage'] = pd.to_numeric(df['Mileage'], errors='coerce')
df['Mileage'] = df['Mileage'].fillna(df['Mileage'].median())


In [10]:
if 'Engine' in df.columns:
    df['Engine'] = df['Engine'].str.replace(" CC","",regex=False).astype(float)

if 'Power' in df.columns:
    df['Power'] = df['Power'].str.replace(" bhp","",regex=False)
    df['Power'] = pd.to_numeric(df['Power'], errors='coerce')
    df['Power'] = df['Power'].fillna(df['Power'].median())

if 'New_Price' in df.columns:
    df['New_Price'] = df['New_Price'].str.replace(" Lakh","",regex=False)
    df['New_Price'] = pd.to_numeric(df['New_Price'], errors='coerce')
    df['New_Price'] = df['New_Price'].fillna(df['New_Price'].median())


In [11]:
cat_cols = ['Fuel_Type','Transmission']
cat_cols = [c for c in cat_cols if c in df.columns]  # avoid errors

df = pd.get_dummies(df, columns=cat_cols, drop_first=True)
df.head()


,Unnamed: 0,Name,Location,Year,Kilometers_Driven,Owner_Type,Mileage,Engine,Power,Seats,New_Price,Price,Fuel_Type_Electric,Fuel_Type_Petrol,Transmission_Manual
0,1,Hyundai Creta 1.6 CRDi SX Option,Pune,2015,41000,First,19.67,1582.0,126.20,5.0,4.78,12.50,False,False,True
1,2,Honda Jazz V,Chennai,2011,46000,First,13.00,1199.0,88.70,5.0,8.61,4.50,False,True,True
2,3,Maruti Ertiga VDI,Chennai,2012,87000,First,20.77,1248.0,88.76,7.0,4.78,6.00,False,False,True
3,4,Audi A4 New 2.0 TDI Multitronic,Coimbatore,2013,40670,Second,15.20,1968.0,140.80,5.0,4.78,17.74,False,False,False
4,6,Nissan Micra Diesel XV,Jaipur,2013,86999,First,23.08,1461.0,63.10,5.0,4.78,3.50,False,False,True


In [12]:
df['Car_Age'] = 2025 - df['Year']
df[['Year','Car_Age']].head()


,Year,Car_Age
0,2015,10
1,2011,14
2,2012,13
3,2013,12
4,2013,12


In [13]:
# Select
df_select = df[['Name','Mileage','Price']].head()

# Filter
df_filter = df[df['Price'] > 10].head()

# Rename
df_rename = df.rename(columns={'Price':'Price_Lakh'}).head()

# Mutate
df['Price_per_CC'] = df['Price'] / df['Engine']

# Arrange
df_sorted = df.sort_values(by='Price', ascending=False).head()

# Summarize
df_summary = df.groupby(df.columns[df.columns.str.contains("Fuel")][0])['Price'].mean()
df_summary


Fuel_Type_Electric
False     9.65264
True     12.87500
Name: Price, dtype: float64

Reporton Findings
1) Missing Values

I checked the dataset and found some missing values.
To avoid losing data, I filled missing numeric values with the median and categorical values with the most common category. This keeps the dataset complete and prevents distortion from extreme values.

2) Removing Units

Several columns had text units like “kmpl”, “CC”, “bhp”, and “Lakh”.
I removed these units and converted everything into numbers so the data can be cleaned, analyzed, and used in calculations.

3) One-Hot Encoding

Fuel type and transmission were text categories, so I converted them into numerical dummy variables.
This makes the dataset usable for statistical analysis and machine learning models.

4) New Feature

I created a new column called Car Age, calculated as 2025 − Year.
Car age helps explain how age affects price and condition.

5) Data Operations

I selected specific columns, filtered high-priced cars, renamed the price column, added a new ratio (Price per CC), sorted cars by price, and calculated average price based on fuel type.